# CIDA-Swarm — базовые линии роя на полной мощности A100

Рой дронов учится искать людей среди зданий: люди бродят, есть приманки, похожие на людей,
глаза ошибаются, в каждом мире своё число дронов (4–8). Две базовые линии, с которыми потом сравним CIDA-связь:

* **nocomm** — каждый дрон сам за себя;
* **stigmergy** — общая карта исследованных клеток (как феромон у муравьёв).

Порядок: **1)** бенчмарк подбирает число миров под вашу карту → **2)** оценка времени → **3)** запуск
(несколько прогонов одновременно, если карта не загружена полностью) → **4)** таблица.
Runtime → Change runtime type → **A100**. Результаты сохраняются на Google Drive.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os
if not os.path.exists("/content/CIDA-plugin"):
    !git clone https://github.com/Kairatzh/CIDA-plugin.git /content/CIDA-plugin
%cd /content/CIDA-plugin
!git pull --ff-only
!git log --oneline -1

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
RUNS = "/content/drive/MyDrive/cida_swarm_runs_v2"
os.makedirs(RUNS, exist_ok=True)

## 1. Бенчмарк (~5–8 минут)

Смотрим, при каком числе миров скорость перестаёт расти, и ускоряет ли `torch.compile`.
`Env only` — скорость одной среды (потолок), `Collect share` — доля времени на сбор опыта;
остальное — обновление сети. Пришлите обе таблицы.

In [ ]:
import sys
!{sys.executable} -u -m cida_swarm.bench --n-envs 8192 16384 32768
# то же с torch.compile: первые итерации компилируются, поэтому iters больше
!{sys.executable} -u -m cida_swarm.bench --n-envs 16384 --iters 8 --extra "--compile"

## 2. Настройки прогона

Впишите `N_ENVS` и `SPEED` из таблицы бенчмарка. `TOTAL_STEPS` — бюджет одного прогона в шагах дронов.
`PARALLEL` — сколько прогонов одновременно: 2 имеет смысл, если пиковая память ≤ 40% карты и загрузка
GPU в `nvidia-smi` ниже ~90%; иначе ставьте 1.

In [ ]:
N_ENVS = 16384        # из бенчмарка
SPEED = 2_000_000     # agent-steps/s из бенчмарка для N_ENVS
TOTAL_STEPS = 3e9     # на один прогон
PARALLEL = 2
SEEDS = [0, 1, 2]

EXPERIMENTS = {
    "nocomm": "",
    "stigmergy": "--share-coverage",
}
COMMON = f"--n-envs {N_ENVS} --total-steps {TOTAL_STEPS:.0f} --eval-every 25"

runs = [(f"{name}_s{seed}", flags, seed) for seed in SEEDS for name, flags in EXPERIMENTS.items()]
todo = [r for r in runs if not os.path.exists(f"{RUNS}/{r[0]}/result.json")]
hours = len(todo) * TOTAL_STEPS / SPEED / 3600 / max(1, PARALLEL) * (1.3 if PARALLEL > 1 else 1.0)
print(f"прогонов осталось: {len(todo)} из {len(runs)}; оценка времени ≈ {hours:.1f} ч")

## 3. Запуск

Прогоны идут фоновыми процессами, логи пишутся в `RUNS/<run>/log.txt`. Ячейка каждые 2 минуты печатает
последнюю строку каждого лога. Если Colab отключился — просто запустите ноутбук заново: готовые прогоны пропустятся.

In [ ]:
import subprocess, time

def tail(path):
    try:
        lines = open(path, encoding="utf-8", errors="ignore").read().strip().splitlines()
        return lines[-1] if lines else "(пусто)"
    except FileNotFoundError:
        return "(ещё не начался)"

queue, running = list(todo), {}
while queue or running:
    while queue and len(running) < PARALLEL:
        run, flags, seed = queue.pop(0)
        out = f"{RUNS}/{run}"
        os.makedirs(out, exist_ok=True)
        cmd = f"{sys.executable} -u -m cida_swarm.train {flags} {COMMON} --seed {seed} --name {run} --out {out}"
        print(">>>", cmd, flush=True)
        running[run] = subprocess.Popen(cmd, shell=True, stdout=open(f"{out}/log.txt", "w"), stderr=subprocess.STDOUT)
    time.sleep(120)
    for run, proc in list(running.items()):
        print(f"[{time.strftime('%H:%M')}] {run}: {tail(f'{RUNS}/{run}/log.txt')}", flush=True)
        if proc.poll() is not None:
            del running[run]
            ok = os.path.exists(f"{RUNS}/{run}/result.json")
            print(f"{'✓' if ok else '✗ ОШИБКА'} {run} завершён" + ("" if ok else " — смотрите log.txt"), flush=True)
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv

## 4. Итоги

In [ ]:
import glob, json, re
from collections import defaultdict
import numpy as np

groups = defaultdict(list)
for path in sorted(glob.glob(f"{RUNS}/*/result.json")):
    r = json.load(open(path))
    groups[re.sub(r"_s\d+$", "", r["name"])].append(r)

def fmt(xs):
    return f"{np.mean(xs):.3f} ± {np.std(xs, ddof=1):.3f}" if len(xs) > 1 else f"{xs[0]:.3f}"

print("| Run | Seeds | Found | All found | Steps | Coverage | Random found | Agent-steps/s |")
print("|---|---|---|---|---|---|---|---|")
for name, runs_ in sorted(groups.items()):
    f = [r["final"] for r in runs_]
    print(f"| {name} | {len(runs_)} | {fmt([x['found_frac'] for x in f])} | {fmt([x['all_found'] for x in f])} "
          f"| {fmt([x['length'] for x in f])} | {fmt([x['coverage'] for x in f])} "
          f"| {fmt([r['random']['found_frac'] for r in runs_])} | {np.mean([r['agent_steps_per_s'] for r in runs_]):,.0f} |")

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, runs_ in sorted(groups.items()):
    color = "C0" if name == "nocomm" else "C1"
    for i, r in enumerate(runs_):
        h = r["history"]
        x = [p["agent_steps"] / 1e9 for p in h]
        axes[0].plot(x, [p["found_frac"] for p in h], color=color, alpha=0.7, label=name if i == 0 else None)
        axes[1].plot(x, [p["all_found"] for p in h], color=color, alpha=0.7, label=name if i == 0 else None)
axes[0].set_title("доля найденных людей"); axes[1].set_title("доля миров, где нашли всех")
for ax in axes:
    ax.set_xlabel("млрд шагов дронов"); ax.grid(alpha=0.3); ax.legend()
plt.show()

## Как читать

* **Found** — доля найденных людей, **All found** — доля миров, где нашли всех, **Coverage** — исследованная
  доля свободной площади.
* Если кривые ещё растут к концу — бюджета мало, увеличьте `TOTAL_STEPS`. Если вышли на плато задолго до конца —
  бюджет можно урезать и сэкономить кредиты.
* Пришлите таблицу бенчмарка и итоговую таблицу: по ним сделаем шаг 2 — связь CIDA.